In [4]:
import os
import google.generativeai as genai
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Configure the Gemini API key
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY not found. Please set it in your .env file.")
genai.configure(api_key=api_key)

def parse_resume_with_gemini(file_path):
    """
    Uploads a resume file (PDF, DOCX, etc.) to the Gemini API and
    prompts the model to extract its text content cleanly.
    
    THIS FUNCTION WORKS FOR .docx, .pdf, .png, .jpg, and more.
    """
    if not os.path.exists(file_path):
        return "Error: File not found."
    
    print(f"Uploading and parsing file with Gemini: {file_path}")
    
    try:
        # Upload the file to the Gemini File API. It handles the format automatically.
        uploaded_file = genai.upload_file(path=file_path)
        
        # Create the prompt for the model
        prompt = """
        You are an expert resume parser. Extract the full text content from the provided resume file.
        Maintain the original structure, spacing, and line breaks as closely as possible.
        Do not add any commentary, just return the extracted text.
        """
        
        # Instantiate the Gemini 1.5 Pro model
        model = genai.GenerativeModel('models/gemini-2.5-pro')
        
        # Generate the content
        response = model.generate_content([prompt, uploaded_file])
        
        # Clean up the uploaded file after we're done
        genai.delete_file(uploaded_file.name)
        
        return response.text

    except Exception as e:
        # Clean up the file even if an error occurs
        if 'uploaded_file' in locals() and uploaded_file:
            try:
                genai.delete_file(uploaded_file.name)
            except Exception as delete_e:
                print(f"Error cleaning up file: {delete_e}")
        return f"An error occurred while parsing with Gemini: {e}"

# --- Main execution block to test with a .docx file ---
if __name__ == "__main__":
    # --- Instructions for testing with DOCX ---
    # 1. Make sure you have your .env file with the GEMINI_API_KEY.
    # 2. Place a sample Word resume (e.g., my_resume.docx) in the 'resumes' folder.
    # 3. Update the `file_to_test` variable below to point to your .docx file.
    
    file_to_test = "D:\Projects\Resume_Builder\deepak_resume.pdf" # <-- CHANGE THIS TO YOUR DOCX FILE
    
    print(f"--- Testing Gemini Parser on DOCX file: {file_to_test} ---")
    
    if os.path.exists(file_to_test):
        extracted_text = parse_resume_with_gemini(file_to_test)
        print("\n--- Extracted Text from DOCX ---")
        print(extracted_text)
        print("\n--- End of Text ---")
    else:
        print(f"\nERROR: The test file was not found at '{file_to_test}'.")
        
    print("\n--- Test Complete ---")

<>:64: SyntaxWarning: invalid escape sequence '\P'
<>:64: SyntaxWarning: invalid escape sequence '\P'
C:\Users\mustangs007\AppData\Local\Temp\ipykernel_106492\4162185007.py:64: SyntaxWarning: invalid escape sequence '\P'
  file_to_test = "D:\Projects\Resume_Builder\deepak_resume.pdf" # <-- CHANGE THIS TO YOUR DOCX FILE


--- Testing Gemini Parser on DOCX file: D:\Projects\Resume_Builder\deepak_resume.pdf ---
Uploading and parsing file with Gemini: D:\Projects\Resume_Builder\deepak_resume.pdf

--- Extracted Text from DOCX ---
DEEPAK KUMAR
Data Scientist – Generative AI, LLM & ML Engineering | Bengaluru, India
Mobile: +91-97806-16787 • deepakkumar007.jobs@gmail.com • linkedin.com/in/mustangs007

| SUMMARY
Full-stack Data Scientist with 5+ years building production ML & GenAl solutions that have saved USD 3.5 M, boosted
revenue 15%, and accelerated workflows up to 6x at Optum, Verizon, and Tiger Analytics. Deep expertise in
Retrieval-Augmented Generation, small-language-model fine-tuning (LoRA, DPO, SFT), time-series forecasting, and MLOps
on Azure.

| CORE COMPETENCIES
• Machine Learning (Regression, Classification, Random Forest, XGBoost, Gradient Boosting, SVM, K-Means, DBSCAN,
  Isolation Forest)
• GenAI & NLP (Llama-3, Phi-3, Mistral, GPT-4, RAG, Vector DB - FAISS/Pinecone, Hugging Face Transformers,

In [12]:
from google import genai
import os
import google.generativeai as genai
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Configure the Gemini API key
api_key = os.getenv("GEMINI_API_KEY")

# Using the genai Client directly
genai.configure(api_key=api_key)

In [14]:
import base64
import logging
import termcolor
from playwright.async_api import async_playwright, Page

logger = logging.getLogger(__name__)

KEY_MAP = {
    "enter": "Enter", "tab": "Tab", "backspace": "BackSpace",
    "escape": "Escape", "delete": "Delete", "space": " ",
    "up": "ArrowUp", "down": "ArrowDown", "left": "ArrowLeft", "right": "ArrowRight",
    "home": "Home", "end": "End", "pageup": "PageUp", "pagedown": "PageDown",
    "ctrl": "Control", "alt": "Alt", "shift": "Shift", "meta": "Meta",
    "f1": "F1", "f2": "F2", "f3": "F3", "f4": "F4", "f5": "F5",
}

class PlaywrightComputer:
    def __init__(
        self,
        screen_size: tuple[int, int] = (1280, 936),
        initial_url: str = "https://www.google.com",
        highlight_mouse: bool = False,
    ):
        self._screen_size = screen_size
        self._initial_url = initial_url
        self._highlight_mouse = highlight_mouse
        self._playwright = None
        self._browser = None
        self._context = None
        self._page = None

    async def __aenter__(self):
        self._playwright = await async_playwright().start()
        width, height = self._screen_size
        self._browser = await self._playwright.chromium.launch(
            headless=False,
            args=[f"--window-size={width},{height}"]
        )
        self._context = await self._browser.new_context(
            viewport={"width": width, "height": height}
        )
        self._page = await self._context.new_page()
        self._context.on("page", self._handle_new_page)
        await self._page.goto(self._initial_url)
        print(termcolor.colored("Browser started.", "green"))
        return self

    async def __aexit__(self, *args):
        await self._context.close()
        await self._browser.close()
        await self._playwright.stop()

    async def _handle_new_page(self, new_page: Page):
        new_url = new_page.url
        await new_page.close()
        await self._page.goto(new_url)

    async def screenshot(self) -> str:
        png_bytes = await self._page.screenshot()
        return base64.b64encode(png_bytes).decode("utf-8")

    async def click(self, x: int, y: int, button: str = "left"):
        await self._page.mouse.click(x, y, button=button)

    async def double_click(self, x: int, y: int):
        await self._page.mouse.dblclick(x, y)

    async def type(self, text: str):
        await self._page.keyboard.type(text)

    async def key(self, key: str):
        keys = [KEY_MAP.get(k.lower(), k) for k in key.split("+")]
        if len(keys) == 1:
            await self._page.keyboard.press(keys[0])
        else:
            for k in keys[:-1]:
                await self._page.keyboard.down(k)
            await self._page.keyboard.press(keys[-1])
            for k in reversed(keys[:-1]):
                await self._page.keyboard.up(k)

    async def move_mouse(self, x: int, y: int):
        await self._page.mouse.move(x, y)

    async def scroll(self, x: int, y: int, delta_x: int, delta_y: int):
        await self._page.mouse.move(x, y)
        await self._page.mouse.wheel(delta_x, delta_y)

    async def navigate(self, url: str):
        await self._page.goto(url)

    def get_current_url(self) -> str:
        return self._page.url

In [15]:
import asyncio
import base64
import termcolor
from google import genai
from google.genai import types

MAX_TURNS = 20
MAX_RECENT_SCREENSHOTS = 3

async def run_agent(query: str, computer, model: str = "gemini-3.1-pro-preview"):
    client = genai.Client()
    width, height = computer._screen_size

    config = types.GenerateContentConfig(
        tools=[types.Tool(
            computer_use=types.ComputerUse(
                environment=types.Environment.ENVIRONMENT_BROWSER
            )
        )],
        thinking_config=types.ThinkingConfig(include_thoughts=True),
    )

    initial_screenshot = await computer.screenshot()
    contents = [
        types.Content(role="user", parts=[
            types.Part(text=query),
            types.Part.from_bytes(
                data=base64.b64decode(initial_screenshot),
                mime_type="image/png"
            )
        ])
    ]

    for turn in range(MAX_TURNS):
        print(termcolor.colored(f"\n--- Turn {turn + 1} ---", "cyan"))

        response = client.models.generate_content(
            model=model,
            contents=contents,
            config=config,
        )

        for part in response.candidates[0].content.parts:
            if hasattr(part, "thought") and part.thought:
                print(termcolor.colored(f"[Thought] {part.text[:200]}...", "yellow"))

        function_calls = [
            p for p in response.candidates[0].content.parts
            if p.function_call is not None
        ]

        if not function_calls:
            final_text = response.text or ""
            print(termcolor.colored(f"\n✅ Done: {final_text}", "green"))
            return final_text

        contents.append(response.candidates[0].content)

        function_responses = []
        for part in function_calls:
            fc = part.function_call
            action = fc.name
            params = dict(fc.args)
            print(termcolor.colored(f"  → {action}({params})", "magenta"))

            result = await handle_action(action, params, computer, width, height)
            function_responses.append(
                types.Part.from_function_response(name=action, response=result)
            )

        await asyncio.sleep(0.5)
        new_screenshot = await computer.screenshot()

        contents.append(types.Content(role="user", parts=function_responses + [
            types.Part.from_bytes(
                data=base64.b64decode(new_screenshot),
                mime_type="image/png"
            )
        ]))

        _prune_screenshots(contents, MAX_RECENT_SCREENSHOTS)

    print(termcolor.colored("⚠️ Reached max turns.", "red"))
    return None


async def handle_action(action: str, params: dict, computer, width: int, height: int) -> dict:
    def dx(x): return int(x * width / 1000)
    def dy(y): return int(y * height / 1000)

    try:
        if action == "click":
            await computer.click(dx(params["x"]), dy(params["y"]), params.get("button", "left"))
            return {"result": "Clicked"}
        elif action == "double_click":
            await computer.double_click(dx(params["x"]), dy(params["y"]))
            return {"result": "Double-clicked"}
        elif action == "type":
            await computer.type(params["text"])
            return {"result": "Typed"}
        elif action == "key":
            await computer.key(params["key"])
            return {"result": "Key pressed"}
        elif action == "move_mouse":
            await computer.move_mouse(dx(params["x"]), dy(params["y"]))
            return {"result": "Mouse moved"}
        elif action == "scroll":
            await computer.scroll(dx(params["x"]), dy(params["y"]),
                                  params.get("delta_x", 0), params.get("delta_y", 0))
            return {"result": "Scrolled"}
        elif action == "navigate":
            await computer.navigate(params["url"])
            return {"result": f"Navigated to {params['url']}"}
        elif action == "wait":
            await asyncio.sleep(params.get("duration", 1))
            return {"result": "Waited"}
        elif action == "get_current_url":
            return {"result": computer.get_current_url()}
        else:
            return {"error": f"Unknown action: {action}"}
    except Exception as e:
        return {"error": str(e)}


def _prune_screenshots(contents: list, keep_last: int):
    screenshot_indices = [
        i for i, c in enumerate(contents)
        if hasattr(c, "parts") and any(
            hasattr(p, "mime_type") and p.mime_type == "image/png"
            for p in c.parts
        )
    ]
    for i in screenshot_indices[:-keep_last]:
        contents[i].parts = [
            p for p in contents[i].parts
            if not (hasattr(p, "mime_type") and p.mime_type == "image/png")
        ]

In [17]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
import sys
import asyncio

if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

import nest_asyncio
nest_asyncio.apply()

async with PlaywrightComputer(
    screen_size=(1280, 936),
    initial_url="https://www.google.com",
) as computer:
    result = await run_agent(
        query="Search for 'Reliance Industries stock price NSE' on Google",
        computer=computer,
    )
    print(result)

NotImplementedError: 

: 